# Dialforge Sales Hard Acceptance v3.3 — Kaggle

This is the current harder marketing gate. It keeps every prior hard case, adds prompt-leakage scoring, negated booking traps, missing-date scheduling, and compound DNC/info cases while retaining the **85 minimum for every shipping model**.

**Before Run All:**
1. Kaggle Settings → Accelerator → NVIDIA GPU. One T4 is enough.
2. Turn **Internet ON**.
3. Click **Run All**.


In [ ]:
import os, pathlib, shutil, subprocess, sys, time, json
ROOT = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path.cwd()
REPO = ROOT / 'Axemetric-Caller-Beta-Runtime'
OUT = ROOT / 'dialforge-hard-sales-v3_3'
print('=== DIALFORGE KAGGLE HARD SALES ACCEPTANCE v3.3 ===')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('No NVIDIA GPU detected. Enable a Kaggle GPU accelerator.')
print('GPU(s):\n' + gpu.stdout.strip())
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_KEEP_ALIVE'] = '30m'
probe = subprocess.run(['curl', '-I', '-L', '--max-time', '15', 'https://github.com'], capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError('Internet appears disabled. Turn Internet ON in Kaggle settings and rerun.')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git', str(REPO)], check=True)
sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Pinned source:', sha)
if shutil.which('zstd') is None:
    subprocess.run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq zstd', shell=True, check=True)
if shutil.which('ollama') is None:
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
log_path = ROOT / 'ollama-hard-sales-v3_3.log'
log = open(log_path, 'w')
ollama = subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
import requests
for _ in range(90):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            break
    except Exception:
        pass
    time.sleep(1)
else:
    log.flush()
    raise RuntimeError('Ollama did not become ready. Log: ' + str(log_path))
print('Ollama ready. Running harder v3.3 gate...')


In [ ]:
import subprocess, json, pathlib, pandas as pd, os, sys
OUT.mkdir(parents=True, exist_ok=True)
runner = REPO / 'benchmarks' / 'dialforge_sales_hard_acceptance_v3_3.py'
cmd = [sys.executable, str(runner), '--output-dir', str(OUT)]
print('Running:', ' '.join(cmd))
proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
lines = []
for line in proc.stdout:
    print(line, end='')
    lines.append(line)
code = proc.wait()
(OUT / 'kaggle-run.log').write_text(''.join(lines), encoding='utf-8')
report_path = OUT / 'dialforge-sales-hard-v3.json'
if not report_path.exists():
    raise RuntimeError(f'No report was produced. Exit code {code}. Check {OUT / "kaggle-run.log"}')
report = json.loads(report_path.read_text(encoding='utf-8'))
rows = []
for model, data in report['models'].items():
    rows.append({'Model':model,'Overall':data['score'],'Sales':data['sales']['score'],'Tools':data['tools']['accuracy'],'Integrity':data['sales']['integrity'],'Critical failures':data['sales']['critical_failures'],'Pass >=85':float(data['score']) >= 85.0})
display(pd.DataFrame(rows))
print('\nFAILED TOOL CASES')
tool_rows=[]
for model,data in report['models'].items():
    for case in data['tools']['cases']:
        if not case['passed']:
            tool_rows.append({'Model':model,'Case':case['id'],'Expected':case['expected'],'Tools':', '.join(case['tools']),'Content':case['content']})
if tool_rows: display(pd.DataFrame(tool_rows))
else: print('None')
print('\nWEAK SALES TURNS (<85 or critical)')
weak=[]
for model,data in report['models'].items():
    for conv in data['sales']['conversations']:
        for turn in conv['turns']:
            if float(turn['score']) < 85 or int(turn['critical']) > 0:
                weak.append({'Model':model,'Conversation':conv['id'],'Expected':turn['expect'],'Score':turn['score'],'Failures':', '.join(turn['failures']),'Answer':turn['answer']})
if weak: display(pd.DataFrame(weak))
else: print('None')
ready = all(float(x['score']) >= 85.0 for x in report['models'].values())
print('\n' + '='*72)
print('MARKETING GATE:', 'PASS' if ready else 'BLOCKED')
print('='*72)
print('Report:', report_path)
print('Log:', OUT / 'kaggle-run.log')
